<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch03_regression_anova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第3回：確率分布と回帰分析の基礎（t検定・ANOVAを線形回帰として）


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Winter & Grawunder (2012), *Journal of Pragmatics*／Winter (2019) 第5-6章／Gries (2013) *Statistics for Linguistics with R*。

In [ ]:
# ---- 共通データセットの読み込み ----
# 第1回と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)

set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5回で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5回以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5回で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3回のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7回のAR(1)を使うときだけ例外）。


■ t検定を回帰として書き直す：

**実際の研究例**：Winter (2019, *Statistics for Linguists*) は、Winter & Grawunder (2012) が収集した韓国語話者の丁寧発話コーパス（フォーマルな場面とカジュアルな場面でのF0）を使い、「2群比較のt検定」と「2群を予測変数に持つ単回帰」が数値として一致することを実演している。本章のコードは、この論証をVOTデータでそのままなぞったものである。こうした「2群比較」は教科書ではt検定として教わることが多いが、実務ではほぼ例外なく回帰として実装される。理由は単純で、

  VOT = β0 + β1・Condition + ε

という線形回帰は、Welchのt検定と（等分散を仮定すれば）同じt値・p値を返す。違いは「モデルとして書けるかどうか」だけで、回帰として書ければ共変量の追加も、後の混合効果モデル（第5回）への拡張も自然に続く（Winter, 2019 第6章）。

In [ ]:
t_result <- t.test(VOT ~ condition, data = vot_data)  # 伝統的なWelchのt検定
lm_result <- lm(VOT ~ condition, data = vot_data)      # 同じ問いを回帰として解く

print(t_result)
summary(lm_result)
# t_resultのt値・p値と、lm_resultの"conditionvoiceless"行のt値・p値が
# ほぼ一致することを確認せよ（自由度の計算法が微妙に違うため完全一致はしない）。

# --- 実行結果（抜粋）---
# 	Welch Two Sample t-test
# t = -28.495, df = 125.23, p-value < 2.2e-16
# sample estimates:
#    mean in group voiced mean in group voiceless
#                22.86689                75.36703
#
# Coefficients:
#                    Estimate Std. Error t value Pr(>|t|)
# (Intercept)          22.867      1.303   17.55   <2e-16 ***
# conditionvoiceless   52.500      1.842   28.50   <2e-16 ***
# Residual standard error: 13.03 on 198 degrees of freedom
# Multiple R-squared:  0.804,	Adjusted R-squared:  0.803
# F-statistic:   812 on 1 and 198 DF,  p-value: < 2.2e-16
#
# 【出力の読み方】
# ・**まず本章の主張を確認する。** t検定のt = -28.495 と、回帰の
#   conditionvoiceless行の t value = 28.50 は、符号を除いてほぼ同じ値である。
#   符号が逆なのは比較の向きの違い（t.test()はvoiced−voiceless、
#   回帰はvoicedを基準にしてvoicelessがどれだけ増えるか）にすぎない。
#   **同じデータに同じ問いを投げれば、名前が違っても同じ答えが返る。**
# ・回帰の (Intercept) 22.867 は「voiced群の平均」。t検定の
#   mean in group voiced 22.86689 と一致する。conditionはfactorで
#   voicedを最初の水準にしてあるので、切片が基準水準の平均になる。
# ・conditionvoiceless の Estimate 52.500 は「voicedからvoicelessへ移ると
#   VOTが何ms増えるか」。実際 75.367 − 22.867 = 52.500 で、
#   t検定のsample estimatesの2つの平均の差にちょうど等しい。
#   **回帰係数は群の差そのもの**であり、難しい変換は要らない。
# ・df（自由度）だけは125.23対198と食い違う。Welchのt検定は2群の分散が
#   違うことを許して自由度を調整するのに対し、lm()は等分散を仮定するため。
#   本書のデータは無声条件のばらつきが大きい（第2回で確認した）ので、
#   この差が出る。**完全一致しないのはこの1点だけ**である。
# ・Multiple R-squared 0.804 は「VOTのばらつきの80.4%を条件だけで説明できる」
#   という意味。残り約20%が話者差・単語差・測定誤差である。
# ・F-statistic 812 は t値の2乗にほぼ等しい（28.50^2 = 812.25）。
#   予測変数が1つのとき、F検定とt検定は同じ検定の別表現になる。
#   これが演習1の答えでもある。
# ・p-value < 2.2e-16 は「2.2×10^-16より小さい」の意味で、Rが表示できる
#   最小値である。**0ではない**し、「効果が巨大」の意味でもない。
#   効果の大きさを述べるときはp値ではなくEstimate（52.5ms）を報告する。


■ ANOVAを回帰として理解する：

genderとconditionの2要因分散分析を、交互作用付き回帰モデルとして推定する。

In [ ]:
aov_result <- aov(VOT ~ condition * gender, data = vot_data)
summary(aov_result)  # F値・p値が要因ごとに出る（伝統的なANOVA表）

lm_result2 <- lm(VOT ~ condition * gender, data = vot_data)
summary(lm_result2)
# aov()のF検定は「回帰係数が複数同時にゼロかどうか」の検定を要約したものに過ぎない。
# lm_result2の係数を見れば、どの水準の組み合わせがどれだけ効いているかまでわかる。

# --- 実行結果（抜粋）---
#                   Df Sum Sq Mean Sq F value   Pr(>F)
# condition          1 137813  137813 877.130  < 2e-16 ***
# gender             1   1027    1027   6.534 0.011341 *
# condition:gender   1   1785    1785  11.360 0.000904 ***
# Residuals        196  30795     157
#
# Coefficients:
#                            Estimate Std. Error t value Pr(>|t|)
# (Intercept)                  23.589      1.773  13.307  < 2e-16 ***
# conditionvoiceless           46.525      2.507  18.559  < 2e-16 ***
# genderM                      -1.444      2.507  -0.576 0.565409
# conditionvoiceless:genderM   11.949      3.545   3.370 0.000904 ***
#
# 【出力の読み方】
# ・ANOVA表の1行が1つの要因。Df=1はどれも2水準（差が1本）だからである。
#   F valueが大きくPr(>F)が小さいほど「その要因は無い」と考えにくい。
# ・**ANOVA表と回帰の係数表が同じことを言っている**のを確かめる。
#   交互作用のPr(>F) 0.000904 と、係数表の
#   conditionvoiceless:genderM の Pr(>|t|) 0.000904 は完全に同じ値である。
#   さらに t = 3.370 を2乗すると 11.357 ≒ F = 11.360。
#   **交互作用のF検定とt検定は、同じ1本の係数を見ている。**
# ・一方 gender は、ANOVA表ではF = 6.534, p = 0.011（有意）なのに、
#   係数表では genderM の p = 0.565（有意でない）と食い違って見える。
#   矛盾ではない。係数表の genderM は「**voiced条件における**MとFの差」
#   （単純主効果）を指すのに対し、ANOVA表のgenderは両条件をまとめた
#   全体の効果を指すためである。treatment codingのままだと、
#   係数表の主効果は基準水準に限った話になる——これを解消する方法が第4回の
#   コントラストコーディングである。
# ・4つの群の平均は係数を足して復元できる。voiced/F = 23.589、
#   voiceless/F = 23.589 + 46.525 = 70.11、voiced/M = 23.589 − 1.444 = 22.15、
#   voiceless/M = 23.589 + 46.525 − 1.444 + 11.949 = 80.62。
#   第1回で作った集計表の4行と一致する。**交互作用項は「4つ目の群だけ
#   足し算からずれる分」を表す。**
#
# ★★ここが本書で最も重要な注意点である。★★
#  この出力は「性別によって有声/無声の作り分けの大きさが違う」（交互作用が
#  有意）と述べているが、**第1回のデータ生成コードにgenderの効果は
#  1行も入れていない。** ではなぜ有意になるのか。
#  S01〜S05をM、S06〜S10をFに割り当てているため、たまたまM側に
#  「大きく作り分ける話者」が多く入った、それだけである。
#  lm()は200行すべてを独立した観測として扱うので、実質10人分しかない
#  情報を200件分の証拠として数えてしまい、**存在しない効果に
#  p = 0.0009という強い有意性を与えている。**
#  これが疑似反復(pseudoreplication)であり、第5回で話者のランダム効果を
#  入れるとこの交互作用は t = 1.14 まで下がって有意でなくなる。
#  **p値が小さいことは、効果が実在することの証明にはならない。**


■ モデル診断：

正規性・等分散性を残差プロットで確認する。

In [ ]:
par(mfrow = c(2, 2))  # 4枚のプロットを2x2で並べる
plot(lm_result2)
# 1枚目 Residuals vs Fitted: 線形性の確認（曲がっていないか）
# 2枚目 Q-Q plot: 残差の正規性の確認（直線に乗っているか）
# 3枚目 Scale-Location: 等分散性の確認（右肩上がりに広がっていないか）
# 4枚目 Residuals vs Leverage: 影響力の強い外れ値がないかの確認

# --- 実行結果 ---
# 図が4枚（2x2）描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・**1枚目 Residuals vs Fitted**（残差 vs 予測値）
#   横軸がモデルの予測値、縦軸が残差（実測−予測）。本書のデータは条件が
#   2値なので、点は左右2本の縦の帯に分かれて並ぶ（連続的な雲にはならない）。
#   赤い線が0の周りで水平なら線形性に問題なし。ここでは水平に近い。
#   ★重要: **左の帯（voicedの予測値付近）より右の帯（voiceless）のほうが
#   縦に長く広がっている。** これが等分散性の崩れであり、第1回で
#   無声条件の残差SDを大きく設定したことがそのまま現れている。
# ・**2枚目 Q-Q Residuals**（正規Q-Qプロット）
#   残差が正規分布に従うなら点が対角線上に乗る。中央付近はよく乗り、
#   両端がわずかに反れる程度なら実用上問題ない。大きくS字に曲がったり、
#   端が大きく跳ねていれば、正規性の仮定か外れ値を疑う。
# ・**3枚目 Scale-Location**
#   縦軸は標準化残差の絶対値の平方根。赤い線が**右肩上がり**なら、
#   予測値が大きいほど残差が大きい＝等分散性が崩れている合図である。
#   このデータでは右肩上がりになる。1枚目で見えたことの確認になっている。
# ・**4枚目 Residuals vs Leverage**
#   leverage（てこ比）は「その観測が回帰線をどれだけ引っ張れるか」。
#   Cookの距離0.5・1.0の破線より外に出た点は、1点だけで結論を変えうる
#   影響力を持つ。ここでは破線自体が図の外にあり、外に出る点は無い。
#   予測変数がカテゴリで各群50件ずつあるため、1点の影響力は原理的に小さい。
# ・**では等分散性が崩れているこのモデルをどう扱うか。**
#   係数の推定値そのものは大きく歪まないが、標準誤差（＝p値と信頼区間）が
#   信頼できなくなる。対処は3つ——(1) 対数変換など応答を変換する、
#   (2) Welchのt検定のように分散を群ごとに許す、(3) 第9回で扱う
#   distributionalモデルで分散自体をモデル化する。
#   **仮定が崩れたまま先へ進み、第5回の混合効果モデルに同じ問題を
#   持ち越すのが最悪の選択である。**


■ 演習：

1. `aov()`と`lm()`の出力から、F値と回帰係数のt値がどう対応するか調べよ。
2. 交互作用項を外したモデルと比較し、`anova(model1, model2)`でモデル比較を行え。

■ 演習の解答：

1問目は「F = t^2」という関係を数値で確かめる問題、2問目は交互作用を入れる価値があるかをモデル比較で判断する問題である。

In [ ]:
# 1. F値と回帰係数のt値の対応を確かめる。
#    予測変数が1つ（2水準）のとき、その要因のF値は係数のt値の2乗に等しい。
m_simple <- lm(VOT ~ condition, data = vot_data)
t_val <- summary(m_simple)$coefficients["conditionvoiceless", "t value"]
f_val <- anova(m_simple)["condition", "F value"]

cat("t値        :", round(t_val, 3), "\n")
cat("t値の2乗   :", round(t_val^2, 3), "\n")
cat("ANOVAのF値 :", round(f_val, 3), "\n")

# 2. 交互作用を入れる意味があるかをモデル比較で判断する。
#    anova(小さいモデル, 大きいモデル)で、増えた項が説明力を有意に上げたかを検定する。
model1 <- lm(VOT ~ condition + gender, data = vot_data)  # 主効果のみ
model2 <- lm(VOT ~ condition * gender, data = vot_data)  # 交互作用あり
anova(model1, model2)

# --- 実行結果 ---
# t値        : 28.495
# t値の2乗   : 811.952
# ANOVAのF値 : 811.952
#
# Analysis of Variance Table
#
# Model 1: VOT ~ condition + gender
# Model 2: VOT ~ condition * gender
#   Res.Df   RSS Df Sum of Sq     F    Pr(>F)
# 1    197 32580
# 2    196 30795  1    1784.9 11.36 0.0009038 ***
#
# 【出力の読み方】
# ・1問目: t値の2乗 811.952 と ANOVAのF値 811.952 が**小数点以下まで完全に
#   一致**する。丸めの範囲でたまたま近いのではなく、数学的に同一である。
#   予測変数が1つ（2水準）のとき、F分布の自由度は(1, n-2)になり、
#   自由度1のF分布は、自由度n-2のt分布の2乗そのものになる。
#   したがって「t検定とANOVAのどちらを使うべきか」という問いは、
#   2群比較の場面では意味を持たない。**同じ検定である。**
# ・2問目: 表の1行が1つのモデル。Res.Dfが残差自由度、RSSが残差平方和
#   （＝モデルで説明できずに残った誤差の大きさ）。交互作用を足すと
#   RSSが32580から30795へ1784.9だけ減った。この「減り」がSum of Sqである。
# ・Df = 1 は、増えた項（交互作用）が係数1本ぶんであることを示す。
#   F = 11.36、Pr(>F) = 0.0009038 は「係数1本の追加でこれだけRSSが減るのは
#   偶然にしては大きい」という判断であり、**交互作用を入れる価値がある**
#   という結論になる。前のセルのANOVA表の交互作用行と同じ値である。
# ・anova()にモデルを2つ渡すときの決まりごとが2つある。
#   (1) **入れ子（nested）でなければならない。** Model 1の項はすべて
#       Model 2に含まれていること。項を入れ替えただけの2モデルは比較できない。
#   (2) **同じデータで当てはめたモデルであること。** 欠損値の扱いで行数が
#       変わっていると、比較そのものが無意味になる。
# ・★ただし、この「交互作用は有意」という結論を鵜呑みにしてはいけない。
#   本文の注意書きのとおり、この交互作用は第1回のデータ生成に入れていない
#   ——話者差を無視したことによる見せかけである。第5回で話者のランダム効果を
#   入れると消える。**モデル比較は「与えられたモデルのどちらが良いか」しか
#   答えず、「両方とも間違っている」可能性は教えてくれない。**
